# 05 独立测试与跨场景泛化评价

完成 E1—E3 并用验证集选定 E3 后，统一评价 **PlantVillage 独立测试集**与**去重后的 PlantDoc 外部集合**。同时分别报告 PlantDoc `train` 来源与 `test` 来源的结果，来源名称不表示本项目训练用途。

默认 `RUN_EVALUATION=False`。评价只用于最终报告，不再依据测试结果调整模型或超参数。指标至少包括 Accuracy、Macro Precision、Macro Recall、Macro-F1、逐类 Precision / Recall / F1 和混淆矩阵，训练耗时读取实际记录。


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        for root in (candidate, candidate / "plant-disease-project"):
            if (root / "configs" / "base.yaml").is_file():
                return root
    raise FileNotFoundError("请在项目根目录或 notebooks 目录打开本 Notebook。")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_config, load_labels

BASE = load_config(PROJECT_ROOT / "configs" / "base.yaml")
EXPERIMENTS = load_config(PROJECT_ROOT / "configs" / "experiments.yaml")
LABELS = load_config(PROJECT_ROOT / "configs" / "labels.yaml")
CLASS_NAMES = [item["label"] for item in load_labels(PROJECT_ROOT)]
assert len(CLASS_NAMES) == 8 and len(set(CLASS_NAMES)) == 8
assert [item["label_index"] for item in LABELS["classes"]] == list(range(8))
print("项目目录：", PROJECT_ROOT)
print("类别数量：", len(CLASS_NAMES), "；随机种子：", BASE["seed"])


## 1. 确认最终模型并运行评价

最终模型由保存的 `selected_runs.e3` 解析，函数重新检查正式完成状态、八类顺序、当前配置和清单指纹。未训练时只提示前置工作。确认后将 `RUN_EVALUATION=True` 并运行本格；不需要手工寻找或填写旧实验目录。


In [ ]:
from src.experiments import selected_run
from src.evaluate import evaluate_final

RUN_EVALUATION = False
final_run = None
try:
    final_run = selected_run(PROJECT_ROOT, "E3")
    print("最终实验：", final_run.name)
except (ValueError, FileNotFoundError) as error:
    print("请完成 02—04 并保存 E3 选择：", error)

if RUN_EVALUATION and final_run is not None:
    evaluate_final(PROJECT_ROOT)
    print("评价结果已保存：", final_run / "metrics.json")
elif not RUN_EVALUATION:
    print("评价关闭：如已完成，可在后续单元加载保存结果。")


## 2. 汇总真实指标与跨场景差值

分别展示 PlantDoc 两个来源及整体结果。来源分组已在整体去重后保留，不能将原始 675 + 69 张直接当作有效评价数。差值定义为“PlantVillage 减 PlantDoc”；类别分布不同、样本量较小和潜在标签噪声都可能影响差值。


In [ ]:
metrics = {}
if final_run is not None and (final_run / "metrics.json").is_file():
    metrics = load_config(final_run / "metrics.json")
pv = metrics.get("plantvillage_test")
pd_external = metrics.get("plantdoc_external_all")
evaluation_sets = {}
if pv:
    evaluation_sets["plantvillage_test"] = pv
if pd_external:
    evaluation_sets["plantdoc_external_all"] = pd_external
    for source_name, source_metrics in pd_external.get("source_splits", {}).items():
        evaluation_sets[f"plantdoc_{source_name}_source"] = source_metrics

summary_rows = []
for dataset_name, result in evaluation_sets.items():
    macro = result["classification_report"]["macro avg"]
    summary_rows.append({"dataset": dataset_name, "samples": result["sample_count"],
                         "accuracy": result["accuracy"], "macro_precision": macro["precision"],
                         "macro_recall": macro["recall"], "macro_f1": result["macro_f1"]})
if not summary_rows:
    print("尚无最终八分类测试结果；不显示旧结果或模拟指标。")
else:
    summary_table = pd.DataFrame(summary_rows)
    display(summary_table)
    table_dir = PROJECT_ROOT / "outputs" / "tables"
    table_dir.mkdir(parents=True, exist_ok=True)
    summary_table.to_csv(table_dir / "cross_domain_comparison.csv", index=False, encoding="utf-8-sig")
    if pv and pd_external:
        print("Accuracy 差值：", pv["accuracy"] - pd_external["accuracy"])
        print("Macro-F1 差值：", pv["macro_f1"] - pd_external["macro_f1"])
    print("训练状态与耗时记录：", final_run / "status.json", final_run / "history.csv")


## 3. 逐类指标与混淆矩阵

混淆矩阵行是真实类别，列是预测类别，编号与标签配置相同。没有真实样本的类别仍保留位置，零支持类别的 Precision / Recall / F1 不能视为稳定能力估计。所有表格、图均来自刚读取的实际指标。


In [ ]:
if not evaluation_sets:
    print("尚无逐类指标。")
else:
    import numpy as np
    figure_dir = PROJECT_ROOT / "outputs" / "figures" / final_run.name
    figure_dir.mkdir(parents=True, exist_ok=True)
    for dataset_name, result in evaluation_sets.items():
        class_rows = [{"label_index": index, "label": label,
                       **result["classification_report"][label]}
                      for index, label in enumerate(CLASS_NAMES)]
        print(dataset_name)
        class_table = pd.DataFrame(class_rows)
        display(class_table)
        class_table.to_csv(PROJECT_ROOT / "outputs" / "tables" / f"{dataset_name}_per_class.csv",
                           index=False, encoding="utf-8-sig")
        matrix = np.asarray(result["confusion_matrix"])
        fig, axis = plt.subplots(figsize=(6, 5))
        plot = axis.imshow(matrix, cmap="Blues")
        axis.set(xticks=range(8), yticks=range(8), xlabel="Predicted class", ylabel="True class",
                 title=dataset_name)
        for row in range(8):
            for column in range(8):
                axis.text(column, row, str(matrix[row, column]), ha="center", va="center", fontsize=8)
        fig.colorbar(plot, ax=axis)
        fig.tight_layout()
        fig.savefig(figure_dir / f"{dataset_name}_confusion_matrix.png", dpi=160)
        plt.show()


## 4. 各类别错误分析

错误表包含每个数据集和真实类别的样本数、错误数、错误率与最常见误判类别；示例来自保存的逐图预测，不据此重新调参。优先结合光照、遮挡、复杂背景、相似病斑和潜在标签错误讨论原因。


In [ ]:
if final_run is None or not (final_run / "predictions.csv").is_file():
    print("尚无逐图预测记录。")
else:
    predictions = pd.read_csv(final_run / "predictions.csv")
    predictions["is_error"] = predictions["true_label"] != predictions["predicted_label"]
    error_rows = []
    for (dataset_name, label), group in predictions.groupby(["dataset", "true_label"], sort=False):
        wrong = group[group["is_error"]]
        error_rows.append({"dataset": dataset_name, "true_label": label, "samples": len(group),
                           "errors": len(wrong), "error_rate": len(wrong) / len(group),
                           "most_confused_with": wrong["predicted_label"].mode().iloc[0] if len(wrong) else ""})
    error_table = pd.DataFrame(error_rows)
    display(error_table)
    error_table.to_csv(PROJECT_ROOT / "outputs" / "tables" / "per_class_error_analysis.csv",
                       index=False, encoding="utf-8-sig")
    display(predictions[predictions["is_error"]].groupby(["dataset", "true_label"], sort=False).head(2))


## 结论记录（评价后填写）

- 最终 run_id、验证选择依据、清单版本、训练耗时与评价日期：待填写。
- 域内和外部整体、两个 PlantDoc 来源的指标与样本数：待填写。
- 各类别表现、主要混淆与差值：待填写。
- 真实场景条件、样本量、类别不平衡、标签映射与来源相关性的限制：待填写。
- 单个种子的单次实验不支持过强泛化结论；测试后发现的优化方向只记录为后续工作。
